# LSTM + Attention Model for Bearing RUL Prediction

This notebook builds and evaluates an **LSTM + Attention** deep learning model for predicting the **Remaining Useful Life (RUL)** of bearings.

## Main objective
The aim is to use engineered vibration features to predict how many time steps remain before bearing failure.

## Why LSTM + Attention?
- **LSTM** learns time-dependent degradation patterns from bearing condition data.
- **Attention** helps the model focus on the most important information for RUL prediction.
- Together, they can improve prediction quality when degradation behaviour changes over time.

## Evaluation metrics
The model performance is checked using:
- **R² score**
- **RMSE**
- **MAE**
- Best prediction
- Worst prediction
- Average error


## Import Required Libraries

This cell imports the libraries required for data processing, scaling, building the LSTM + Attention model, evaluation, and visualization.

In [15]:
import os
import pandas as pd
import numpy as np
from scipy.stats import kurtosis, skew
from scipy.signal import welch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Attention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded!")

Libraries loaded!


## Load Engineered Feature Dataset

This cell loads the processed bearing dataset containing vibration features and RUL labels.

In [16]:
df = pd.read_csv("new_features_hanna.csv")

print(df.shape)
print(df.columns)
print(df.head())

(7534, 13)
Index(['rms_x', 'kurtosis_x', 'crest_x', 'spectral_entropy_x',
       'fft_band_energy_x', 'rms_y', 'kurtosis_y', 'crest_y',
       'spectral_entropy_y', 'fft_band_energy_y', 'RUL', 'bearing',
       'time_step'],
      dtype='object')
      rms_x  kurtosis_x   crest_x  spectral_entropy_x  fft_band_energy_x  \
0  0.561746   -0.131465  3.578132            0.737411      881481.869549   
1  0.535112   -0.084646  3.578687            0.771654      814291.485204   
2  0.531158    0.033388  3.578971            0.756740      788306.889011   
3  0.554833    0.043419  3.442476            0.747789      862229.478882   
4  0.566652   -0.185477  3.118317            0.742210      886410.660726   

      rms_y  kurtosis_y   crest_y  spectral_entropy_y  fft_band_energy_y  \
0  0.435801   -0.035080  3.650745            0.889255      218045.175171   
1  0.420968    0.150620  3.957542            0.897493      218825.543172   
2  0.425605   -0.061552  3.721758            0.887962      187232.91

In [17]:
df = df.sort_values(["bearing", "time_step"]).reset_index(drop=True)

df["total_steps"] = df.groupby("bearing")["time_step"].transform("max")
df["RUL_norm"] = df["RUL"] / df["total_steps"]
df["life_fraction"] = df["time_step"] / df["total_steps"]

print(df[["bearing", "time_step", "RUL", "total_steps", "RUL_norm"]].head())

      bearing  time_step   RUL  total_steps  RUL_norm
0  Bearing1_1          0  2802         2802  1.000000
1  Bearing1_1          1  2801         2802  0.999643
2  Bearing1_1          2  2800         2802  0.999286
3  Bearing1_1          3  2799         2802  0.998929
4  Bearing1_1          4  2798         2802  0.998572


## Select Features and Target

This cell selects the input vibration features and defines **RUL** as the target variable.

In [18]:
feature_cols = [
    "rms_x",
    "kurtosis_x",
    "crest_x",
    "spectral_entropy_x",
    "fft_band_energy_x",
    "rms_y",
    "kurtosis_y",
    "crest_y"
]

target_col = "RUL_norm"

In [19]:
print(df.columns.tolist())

['rms_x', 'kurtosis_x', 'crest_x', 'spectral_entropy_x', 'fft_band_energy_x', 'rms_y', 'kurtosis_y', 'crest_y', 'spectral_entropy_y', 'fft_band_energy_y', 'RUL', 'bearing', 'time_step', 'total_steps', 'RUL_norm', 'life_fraction']


## Split Bearings into Training and Validation Sets

This section splits the dataset based on entire bearing runs instead of random rows.

### Why GroupShuffleSplit?
In bearing RUL prediction, samples from the same bearing are highly related.
If random splitting is used, the model may see information from the same bearing in both training and validation data, causing data leakage.

To avoid this:
- Entire bearings are separated into either training or validation sets
- No overlap exists between bearings
- The model is evaluated on completely unseen bearings

### Split Details
- Training set: 67%
- Validation set: 33%
- Random state: 42 (for reproducibility)

The overlap check confirms that no bearing appears in both datasets.

In [20]:
X = df[feature_cols].values
y = df[target_col].values
groups = df["bearing"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

print("Train bearings:", train_df["bearing"].unique())
print("Val bearings:", val_df["bearing"].unique())
print("Overlap:", set(train_df["bearing"]) & set(val_df["bearing"]))

Train bearings: ['Bearing2_1' 'Bearing2_2' 'Bearing3_1' 'Bearing3_2']
Val bearings: ['Bearing1_1' 'Bearing1_2']
Overlap: set()


## Scale Features and Target Values

Feature scaling is applied before training the LSTM + Attention model.

### Why scaling is important
Deep learning models perform better when input features are normalized or standardized.

In this step:
- `StandardScaler` is used for input features
- `MinMaxScaler` is used for the target RUL values

This helps:
- Faster model convergence
- Stable gradient updates
- Improved prediction performance

In [21]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

feature_scaler = StandardScaler()
target_scaler = MinMaxScaler()

train_df[feature_cols] = feature_scaler.fit_transform(train_df[feature_cols])
val_df[feature_cols] = feature_scaler.transform(val_df[feature_cols])

train_df[[target_col]] = target_scaler.fit_transform(train_df[[target_col]])
val_df[[target_col]] = target_scaler.transform(val_df[[target_col]])

print("Scaling completed")

Scaling completed


In [22]:
def create_sequences(data, feature_cols, target_col, seq_length=20):
    X_seq = []
    y_seq = []

    for bearing in data["bearing"].unique():
        bearing_df = data[data["bearing"] == bearing].sort_values("time_step")

        X_values = bearing_df[feature_cols].values
        y_values = bearing_df[target_col].values

        for i in range(len(bearing_df) - seq_length):
            X_seq.append(X_values[i:i + seq_length])
            y_seq.append(y_values[i + seq_length])

    return np.array(X_seq), np.array(y_seq)

In [23]:
SEQ_LENGTH = 20

X_train_seq, y_train_seq = create_sequences(
    train_df,
    feature_cols,
    target_col,
    SEQ_LENGTH
)

X_val_seq, y_val_seq = create_sequences(
    val_df,
    feature_cols,
    target_col,
    SEQ_LENGTH
)

print("X_train_seq:", X_train_seq.shape)
print("y_train_seq:", y_train_seq.shape)
print("X_val_seq:", X_val_seq.shape)
print("y_val_seq:", y_val_seq.shape)

X_train_seq: (3780, 20, 8)
y_train_seq: (3780,)
X_val_seq: (3634, 20, 8)
y_val_seq: (3634,)


## Build the LSTM + Attention Model

This section defines the deep learning architecture used for Remaining Useful Life (RUL) prediction.

In [24]:
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Attention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

inputs = Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2]))

x = LSTM(64, return_sequences=True)(inputs)
x = Dropout(0.2)(x)

attention = Attention()([x, x])
x = GlobalAveragePooling1D()(attention)

x = Dense(32, activation="relu")(x)
x = Dropout(0.2)(x)

outputs = Dense(1)(x)

lstm_attention_model = Model(inputs, outputs)

lstm_attention_model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="mse",
    metrics=["mae"]
)

lstm_attention_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 20, 8)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 20, 64)    │     18,688 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 20, 64)    │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_1         │ (None, 20, 64)    │          0 │ dropout_2[0][0],  │
│ (Attention)         │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ attention_1[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,801 (81.25 KB)

 Trainable params: 20,801 (81.25 KB)

 Non-trainable params: 0 (0.00 B)

## Train the LSTM + Attention Model

This section trains the LSTM + Attention network using the prepared sequence data.

In [25]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=8,
    min_lr=1e-6
)

history = lstm_attention_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=150,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


Epoch 1/150
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.1092 - mae: 0.2567 - val_loss: 0.0325 - val_mae: 0.1512 - learning_rate: 5.0000e-04
Epoch 2/150
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0423 - mae: 0.1586 - val_loss: 0.0759 - val_mae: 0.2394 - learning_rate: 5.0000e-04
Epoch 3/150
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0312 - mae: 0.1357 - val_loss: 0.1396 - val_mae: 0.3135 - learning_rate: 5.0000e-04
Epoch 4/150
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0249 - mae: 0.1212 - val_loss: 0.1615 - val_mae: 0.3394 - learning_rate: 5.0000e-04
Epoch 5/150
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0225 - mae: 0.1157 - val_loss: 0.1455 - val_mae: 0.3202 - learning_rate: 5.0000e-04
Epoch 6/150
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0207 - mae: 0.1118 - val_loss: 0.1260 - val_mae: 0.2979 - learning_rate: 5.0000e-04
Epoch 7/150
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0191 - mae: 0.1063 - val_loss: 0.1408 - val_mae: 0.31

## Evaluate LSTM + Attention Model Performance

This section evaluates the trained LSTM + Attention model using validation data.

In [26]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

y_pred_scaled = lstm_attention_model.predict(X_val_seq)

y_pred = target_scaler.inverse_transform(
    y_pred_scaled.reshape(-1, 1)
).flatten()

y_true = target_scaler.inverse_transform(
    y_val_seq.reshape(-1, 1)
).flatten()

r2 = r2_score(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)

print("LSTM + Attention Results")
print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
LSTM + Attention Results
R²   : 0.6016
RMSE : 0.1803
MAE  : 0.1512


In [27]:
import joblib

lstm_attention_model.save("Hanna_LSTM_Attention_8features_model.keras")

joblib.dump(feature_scaler, "Hanna_LSTM_Attention_8features_feature_scaler.pkl")
joblib.dump(target_scaler, "Hanna_LSTM_Attention_8features_target_scaler.pkl")
joblib.dump(feature_cols, "Hanna_LSTM_Attention_8features_feature_cols.pkl")

print("Model, scalers, and feature list saved.")

Model, scalers, and feature list saved.


Load Unseen Test Bearings

The unseen `Test_set` bearings are loaded and processed using the same vibration feature extraction pipeline used during training.

These bearings were not used during model validation or training.

Extract Statistical Vibration Features

Statistical vibration features are extracted from the raw accelerometer signals in both x and y directions.

Extracted features include:

- RMS
- Kurtosis
- Crest factor
- Spectral entropy
- FFT band energy

These features are extracted from x and y vibration directions where applicable and characterize bearing degradation behavior for predictive maintenance analysis.

In [28]:
import os
import pandas as pd
import numpy as np
from scipy.stats import kurtosis
from scipy.signal import welch

def spectral_entropy(signal):
    freqs, psd = welch(signal)
    psd_norm = psd / (np.sum(psd) + 1e-12)
    return -np.sum(psd_norm * np.log(psd_norm + 1e-12))

def fft_band_energy(signal):
    freqs, psd = welch(signal)
    return np.sum(psd)

def extract_features(filepath):
    df_raw = pd.read_csv(filepath, header=None)

    ax = df_raw[4].values
    ay = df_raw[5].values if 5 in df_raw.columns else ax

    features = {}

    features["rms_x"] = np.sqrt(np.mean(ax**2))
    features["kurtosis_x"] = kurtosis(ax)
    features["crest_x"] = np.max(np.abs(ax)) / (features["rms_x"] + 1e-10)
    features["spectral_entropy_x"] = spectral_entropy(ax)
    features["fft_band_energy_x"] = fft_band_energy(ax)

    features["rms_y"] = np.sqrt(np.mean(ay**2))
    features["kurtosis_y"] = kurtosis(ay)
    features["crest_y"] = np.max(np.abs(ay)) / (features["rms_y"] + 1e-10)

    return features


test_path = "Test_set"
all_test = []

for bearing in sorted(os.listdir(test_path)):
    path = os.path.join(test_path, bearing)

    if not os.path.isdir(path):
        continue

    files = sorted([f for f in os.listdir(path) if f.startswith("acc_")])

    records = []

    for i, f in enumerate(files):
        feats = extract_features(os.path.join(path, f))

        feats["time_step"] = i
        feats["total_steps"] = len(files)
        feats["bearing"] = bearing

        records.append(feats)

    all_test.append(pd.DataFrame(records))
    print(f"{bearing}: {len(files)} files")

test_df = pd.concat(all_test, ignore_index=True)

print(f"\nTotal test rows: {test_df.shape[0]}")
print(test_df.head())

Bearing1_3: 1802 files
Bearing1_4: 1139 files
Bearing1_5: 2302 files
Bearing1_6: 2302 files
Bearing1_7: 1502 files
Bearing2_3: 1202 files
Bearing2_4: 612 files
Bearing2_5: 2002 files
Bearing2_6: 572 files
Bearing2_7: 172 files
Bearing3_3: 352 files

Total test rows: 13959
      rms_x  kurtosis_x   crest_x  spectral_entropy_x  fft_band_energy_x  \
0  0.415616    0.068604  3.556165            3.939009          44.428896   
1  0.391144    0.196395  3.868142            4.081279          38.243359   
2  0.389165    0.380518  4.116510            4.083049          38.107697   
3  0.380670    0.489310  4.266161            4.029995          37.119444   
4  0.400809    0.268491  3.979449            4.047669          39.764721   

      rms_y  kurtosis_y   crest_y  time_step  total_steps     bearing  
0  0.302195    0.045011  3.577161          0         1802  Bearing1_3  
1  0.305312   -0.012812  3.629077          1         1802  Bearing1_3  
2  0.301042    0.109585  4.009405          2         1

In [29]:
print("Unseen Test_set bearings:")
print(test_df["bearing"].unique())

print("\nTraining bearings:")
print(train_df["bearing"].unique())

print("\nValidation bearings:")
print(val_df["bearing"].unique())

print("\nOverlap with training:")
print(set(test_df["bearing"].unique()) & set(train_df["bearing"].unique()))

print("\nOverlap with validation:")
print(set(test_df["bearing"].unique()) & set(val_df["bearing"].unique()))

Unseen Test_set bearings:
['Bearing1_3' 'Bearing1_4' 'Bearing1_5' 'Bearing1_6' 'Bearing1_7'
 'Bearing2_3' 'Bearing2_4' 'Bearing2_5' 'Bearing2_6' 'Bearing2_7'
 'Bearing3_3']

Training bearings:
['Bearing2_1' 'Bearing2_2' 'Bearing3_1' 'Bearing3_2']

Validation bearings:
['Bearing1_1' 'Bearing1_2']

Overlap with training:
set()

Overlap with validation:
set()


Predict Remaining Useful Life on Unseen Bearings

The final trained LSTM + Attention model predicts the normalized Remaining Useful Life (RUL) for unseen bearings from the `Test_set` dataset.

The continuous RUL predictions are then smoothed for maintenance interpretation.

In [30]:
test_df = test_df.sort_values(["bearing", "time_step"]).reset_index(drop=True)

test_df["RUL"] = test_df["total_steps"] - test_df["time_step"]
test_df["RUL_norm"] = test_df["RUL"] / test_df["total_steps"]

test_df[feature_cols] = feature_scaler.transform(test_df[feature_cols])

print(test_df[["bearing", "time_step", "RUL", "RUL_norm"]].head())

      bearing  time_step   RUL  RUL_norm
0  Bearing1_3          0  1802  1.000000
1  Bearing1_3          1  1801  0.999445
2  Bearing1_3          2  1800  0.998890
3  Bearing1_3          3  1799  0.998335
4  Bearing1_3          4  1798  0.997780


In [31]:
def create_test_sequences(data, feature_cols, seq_length=20):
    X_seq = []
    meta = []

    for bearing in data["bearing"].unique():
        bearing_df = data[data["bearing"] == bearing].sort_values("time_step")

        X_values = bearing_df[feature_cols].values

        for i in range(len(bearing_df) - seq_length):
            X_seq.append(X_values[i:i + seq_length])

            meta.append({
                "bearing": bearing,
                "time_step": bearing_df.iloc[i + seq_length]["time_step"],
                "total_steps": bearing_df.iloc[i + seq_length]["total_steps"],
                "Actual_RUL_norm": bearing_df.iloc[i + seq_length]["RUL_norm"]
            })

    return np.array(X_seq), pd.DataFrame(meta)


X_test_seq, test_meta = create_test_sequences(
    test_df,
    feature_cols,
    SEQ_LENGTH
)

print("X_test_seq shape:", X_test_seq.shape)
print(test_meta.head())

X_test_seq shape: (13739, 20, 8)
      bearing  time_step  total_steps  Actual_RUL_norm
0  Bearing1_3         20         1802         0.988901
1  Bearing1_3         21         1802         0.988346
2  Bearing1_3         22         1802         0.987791
3  Bearing1_3         23         1802         0.987236
4  Bearing1_3         24         1802         0.986681


In [32]:
test_pred_scaled = lstm_attention_model.predict(X_test_seq)

test_pred_norm = target_scaler.inverse_transform(
    test_pred_scaled.reshape(-1, 1)
).flatten()

test_results = test_meta.copy()
test_results["pred_norm"] = np.clip(test_pred_norm, 0, 1)

print(test_results.head())

430/430 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
      bearing  time_step  total_steps  Actual_RUL_norm  pred_norm
0  Bearing1_3         20         1802         0.988901   0.142403
1  Bearing1_3         21         1802         0.988346   0.142316
2  Bearing1_3         22         1802         0.987791   0.142581
3  Bearing1_3         23         1802         0.987236   0.142262
4  Bearing1_3         24         1802         0.986681   0.142303


In [33]:
smoothed_pred = []
smoothed_std = []

for bearing in test_results["bearing"].unique():
    mask = test_results["bearing"] == bearing

    preds = pd.Series(test_results.loc[mask, "pred_norm"])

    smooth = preds.rolling(20, min_periods=1).mean()
    std = preds.rolling(20, min_periods=1).std().fillna(0)

    smoothed_pred.extend(smooth.values)
    smoothed_std.extend(std.values)

test_results["pred_norm_smooth"] = np.clip(smoothed_pred, 0, 1)
test_results["std_norm"] = smoothed_std

test_results["Predicted_RUL_s"] = (
    test_results["pred_norm_smooth"] * test_results["total_steps"] * 10
)

test_results["Uncertainty_s"] = (
    test_results["std_norm"] * test_results["total_steps"] * 10
)

test_results["Lower_RUL_s"] = np.maximum(
    test_results["Predicted_RUL_s"] - test_results["Uncertainty_s"],
    0
)

test_results["Upper_RUL_s"] = (
    test_results["Predicted_RUL_s"] + test_results["Uncertainty_s"]
)

print("Predictions done!")

Predictions done!


Health-State Classification

The predicted Remaining Useful Life values are converted into maintenance-oriented health states:

- Non-critical
- Wear detectable
- Imminent failure

This converts continuous RUL predictions into interpretable maintenance decisions for predictive maintenance applications.

In [34]:
def classify_health(rul_norm):
    if rul_norm <= 0.20:
        return "Imminent failure"
    elif rul_norm <= 0.50:
        return "Wear detectable"
    else:
        return "Non-critical"


test_results["Health_State"] = test_results["pred_norm_smooth"].apply(classify_health)

print(test_results.head())

      bearing  time_step  total_steps  Actual_RUL_norm  pred_norm  \
0  Bearing1_3         20         1802         0.988901   0.142403   
1  Bearing1_3         21         1802         0.988346   0.142316   
2  Bearing1_3         22         1802         0.987791   0.142581   
3  Bearing1_3         23         1802         0.987236   0.142262   
4  Bearing1_3         24         1802         0.986681   0.142303   

   pred_norm_smooth  std_norm  Predicted_RUL_s  Uncertainty_s  Lower_RUL_s  \
0          0.142403  0.000000      2566.101104       0.000000  2566.101104   
1          0.142360  0.000061      2565.319311       1.105622  2564.213689   
2          0.142433  0.000135      2566.646824       2.428594  2564.218230   
3          0.142390  0.000139      2565.875884       2.511860  2563.364023   
4          0.142373  0.000127      2565.559716       2.287333  2563.272383   

   Upper_RUL_s      Health_State  
0  2566.101104  Imminent failure  
1  2566.424933  Imminent failure  
2  2569.075

Final Bearing-Level Evaluation

The remaining useful life generated from:

`total_steps - time_step`

is used only for sequence preparation and degradation trend representation inside the available Test_set recordings.

For final predictive maintenance evaluation, the official PRONOSTIA ground-truth RUL values are used for each unseen test bearing.

In [35]:
actual_rul = {
    "Bearing1_3": 5730,
    "Bearing1_4": 339,
    "Bearing1_5": 1610,
    "Bearing1_6": 1460,
    "Bearing1_7": 7570,
    "Bearing2_3": 7530,
    "Bearing2_4": 1390,
    "Bearing2_5": 3090,
    "Bearing2_6": 1290,
    "Bearing2_7": 580,
    "Bearing3_3": 820
}

latest = (
    test_results
    .sort_values("time_step")
    .groupby("bearing")
    .last()
    .reset_index()
)

print("Best model: LSTM + Attention\n")
print(f"{'Bearing':<12} {'Predicted(s)':>13} {'Actual(s)':>10} {'Error%':>8} {'Uncertainty':>13} {'Health'}")
print("-" * 90)

errors = []

for _, row in latest.iterrows():
    b = row["bearing"]
    pred_s = row["Predicted_RUL_s"]
    act_s = actual_rul[b]
    unc_s = row["Uncertainty_s"]
    health = row["Health_State"]

    err = abs(pred_s - act_s) / act_s * 100
    errors.append(err)

    print(f"{b:<12} {pred_s:>13.0f} {act_s:>10} {err:>7.1f}% {unc_s:>12.0f}s  {health}")

print("-" * 90)
print(f"Average Error: {np.mean(errors):.1f}%")
print(f"Best  Error:   {np.min(errors):.1f}%")
print(f"Worst Error:   {np.max(errors):.1f}%")

Best model: LSTM + Attention

Bearing       Predicted(s)  Actual(s)   Error%   Uncertainty Health
------------------------------------------------------------------------------------------
Bearing1_3            3784       5730    34.0%          141s  Wear detectable
Bearing1_4            2131        339   528.6%          215s  Imminent failure
Bearing1_5            3313       1610   105.7%           20s  Imminent failure
Bearing1_6            3009       1460   106.1%           23s  Imminent failure
Bearing1_7            2032       7570    73.2%            5s  Imminent failure
Bearing2_3            1636       7530    78.3%           63s  Imminent failure
Bearing2_4             857       1390    38.3%            5s  Imminent failure
Bearing2_5            2827       3090     8.5%            6s  Imminent failure
Bearing2_6             815       1290    36.9%            2s  Imminent failure
Bearing2_7             223        580    61.6%            8s  Imminent failure
Bearing3_3            

Save Predictive Maintenance Results

The final prediction results for unseen bearings are saved as CSV files containing:

- predicted RUL
- uncertainty estimates
- confidence bounds
- health-state classifications

These outputs support condition-based maintenance planning and failure prevention.

In [36]:
test_results.to_csv(
    "Hanna_LSTM_Attention_test_predictionsold_timeseries.csv",
    index=False
)

latest.to_csv(
    "Hanna_LSTM_Attention_test_predictionsold_final.csv",
    index=False
)

print("Saved: Hanna_LSTM_Attention_test_predictionsold_timeseries.csv")
print("Saved: Hanna_LSTM_Attention_test_predictionsold_final.csv")
print("\nModelling complete! Best model: LSTM + Attention")

Saved: Hanna_LSTM_Attention_test_predictionsold_timeseries.csv
Saved: Hanna_LSTM_Attention_test_predictionsold_final.csv

Modelling complete! Best model: LSTM + Attention
